## LOADING DOCUMENT

In [2]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("consumer_act.pdf")
documents = loader.load()

## CHUNKING

In [3]:
#chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter 
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=300)
chunks=text_splitter.split_documents(documents)



In [24]:
for chunk in chunks[:2]:
    print(chunk.metadata)

{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-10-29T11:22:26+05:30', 'author': 'Lenovo', 'moddate': '2024-10-29T11:22:26+05:30', 'source': 'consumer_act.pdf', 'total_pages': 39, 'page': 0, 'page_label': '1'}
{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-10-29T11:22:26+05:30', 'author': 'Lenovo', 'moddate': '2024-10-29T11:22:26+05:30', 'source': 'consumer_act.pdf', 'total_pages': 39, 'page': 0, 'page_label': '1'}


## EMBEDDING

In [18]:
from langchain_core.vectorstores import InMemoryVectorStore
import google.generativeai as genai

text=[doc.page_content for doc in chunks]

from langchain_huggingface import HuggingFaceEmbeddings  

#wrapped senttransformer inside huggingface bcoz vector store doesn't store numpy array and sentantranformer generate numpy array 

model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
vectorstore=InMemoryVectorStore(embedding=model)
vectorstore.add_documents(documents=chunks)

['ca45d616-2b1c-4f02-878f-112d86066944',
 '7e0eb1db-f45a-4418-8b00-33b418b61eda',
 '0cc5e5f0-5639-41aa-85d8-ad3312416f42',
 '04d26f83-3b83-4a8e-93d9-5a65c4b3c075',
 'e415f505-b7fc-4155-b993-08b99f35cc6e',
 '2130cea0-d088-47d6-810a-625b91ca7b99',
 'b9b1a397-dd56-4f01-bdd8-0395d476d414',
 '9e657ae5-2657-4113-a32a-83a44b66c2c6',
 '0d72b7a6-9f8d-446e-a7b9-39879583fdde',
 '1df4c001-e7af-44ab-97d8-b44d565c61e4',
 '6c9f473a-be4b-4e15-9cca-43fe04ef632a',
 '22091505-f202-4f9c-bcd0-bdd4b28f3bcf',
 '77f33d91-c2f1-4ffb-ac9f-2ce748c68fce',
 '45ee0a9d-4774-4a9c-9d67-72c025cce136',
 'a43f3c22-6dfb-437b-92a6-5153f24e7d79',
 '353c3cb1-f622-4fd7-9f52-54e966040c3e',
 '8442e72f-1b3a-4343-9884-f3685f158d71',
 '2a9bf55a-ac85-427b-85ad-c617257094eb',
 '50ab915a-3bd0-47c2-96a2-9e1235c1ec0a',
 'e94da673-2175-4058-8321-4400967c2074',
 '476cc6cb-874a-498e-8150-56c2b079cfd0',
 '86f83fda-0bab-4652-a2c6-be89e7a0205a',
 'c5535281-c22f-459c-b7ac-c792ac2bdbf7',
 'd78e21a8-681a-4d56-bbeb-489766b8e686',
 'f2bdc832-3fd7-

# creating prompt template

In [21]:
question="I purchased a Samsung Galaxy A35 5G from XYZ Electronics Store on 10 June 2026. Within one week of using the phone, it started overheating during normal use. The battery also drains very quickly, and the phone restarts automatically several times a day."
retriever = vectorstore.as_retriever()
retrieved_documents = retriever.invoke(question)

len(retrieved_documents)

4

In [38]:
#loading api key
import os
import dotenv
from dotenv import load_dotenv
load_dotenv('.env.txt')


True

In [25]:
import google.generativeai as genai
genai.configure(api_key="AQ.Ab8RN6LCunj0EYU2BkbLyo2VUGYB73tu2Elw9DZ3aofrabj-hQ")
llm = genai.GenerativeModel("gemini-2.5-flash")
response = llm.generate_content(
    f"""You are an AI legal assistant for the Consumer Protection Act, 2019.

Instructions:
1. Answer ONLY using the retrieved context.
2. First identify the consumer's issue.
3. Explain how the retrieved legal provisions apply to the user's situation.
4. Mention the relevant sections used.
5. If the context supports it, explain the remedies available.
6. Do not include unrelated provisions.
7. If the context is insufficient, explicitly state that additional legal provisions are needed.
8. Never invent laws or sections.
    retrieved context:{retrieved_documents}
    questions:{question}

Your citations MUST include:
- Document name
- Page number
- Section/article if available
- Source/file name

Citation format:

[Source: <document name>, Page: <page_label>, Section: <section>]

If a particular metadata field is unavailable, write "Not available".
DO NOT invent page numbers, sections, document names, or other metadata.

If multiple pieces of information come from different pages, cite each relevant page separately.
"""


)
print(response.text)

The consumer's issue is that their newly purchased Samsung Galaxy A35 5G phone is defective, exhibiting issues such as overheating, rapid battery drain, and automatic restarts within one week of use.

The retrieved legal provisions from the Consumer Protection Act, 2019, apply to your situation under the Chapter on Product Liability.

1.  **Application of Product Liability:** Your claim for harm caused by the defective phone falls under a "product liability action" because the phone was manufactured, sold, and likely serviced.
    [Source: consumer_act.pdf, Page: 31, Section: 82]
2.  **Right to Bring Action:** You, as a complainant, can bring a product liability action against the product manufacturer (Samsung), product service provider, or product seller (XYZ Electronics Store) for the harm caused by the defective phone.
    [Source: consumer_act.pdf, Page: 31, Section: 83]
3.  **Manufacturer's Liability:** A product manufacturer is liable in a product liability action if:
    *   The